In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
# ── Step 1: Define Dataset Paths ─────────────────────────────────────────────
# Images are organised as: path/class_name/image.jpg
train_path = "Datasets/Assignment_3/train"
valid_path = "Datasets/Assignment_3/valid"
test_path  = "Datasets/Assignment_3/test"

In [ ]:
# ── Step 2: Image Preprocessing (Data Augmentation) ──────────────────────────
# Training: augment images so the model sees more variety → less overfitting
#   rescale: pixel values 0-255 → 0-1 (normalise)
#   rotation_range, zoom_range, horizontal_flip: random transformations
# Validation/Test: only rescale, NO augmentation (we want real performance numbers)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)
valid_datagen = ImageDataGenerator(rescale=1./255)
test_datagen  = ImageDataGenerator(rescale=1./255)

In [ ]:
# ── Step 3: Load Images from Folders ─────────────────────────────────────────
# flow_from_directory: reads images from subfolders, resizes to 128×128
# class_mode='categorical': one-hot encodes labels for multi-class classification
# test: class_mode=None + shuffle=False → predictions stay in order
train_data = train_datagen.flow_from_directory(
    train_path, target_size=(128, 128), batch_size=32, class_mode='categorical'
)
valid_data = valid_datagen.flow_from_directory(
    valid_path, target_size=(128, 128), batch_size=32, class_mode='categorical'
)
test_data = test_datagen.flow_from_directory(
    test_path, target_size=(128, 128), batch_size=16, class_mode=None, shuffle=False
)

In [ ]:
# ── Step 4: Inspect Class Labels ─────────────────────────────────────────────
# 38 classes: each is a plant species + disease (or healthy)
print("Number of classes:", train_data.num_classes)
print("\nClass Labels:")
print(train_data.class_indices)

In [ ]:
# ── Step 5: Build CNN — Convolutional Blocks ─────────────────────────────────
# Each block: Conv2D (detect features) → MaxPooling (reduce size) → Dropout (regularise)
# Filters double each block (32 → 64 → 128): learns increasingly complex features
model = Sequential()

# Block 1: detects simple edges and textures
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# Block 2: detects shapes and patterns
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# Block 3: detects high-level disease features
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

In [ ]:
# ── Step 5 (cont): Classifier Head ───────────────────────────────────────────
# Flatten: converts 3D feature maps → 1D vector for Dense layers
# Dense(128): fully connected layer to combine all features
# Dropout(0.5): aggressive regularisation before the final classification
# softmax: outputs a probability for each of the 38 classes (sum = 1)
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(train_data.num_classes, activation='softmax'))

In [ ]:
# ── Step 6: Compile ───────────────────────────────────────────────────────────
# categorical_crossentropy: correct loss for multi-class with softmax output
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# ── Step 7: Train ────────────────────────────────────────────────────────────
# validation_data: uses a separate generator (not a split) for honest validation
history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=10
)

In [ ]:
# ── Step 8: Save Model ────────────────────────────────────────────────────────
# .keras format is the modern Keras standard (no deprecation warnings)
model.save("plant_disease_cnn_model.keras")
print("Model saved successfully!")

In [ ]:
# ── Step 9: Print Final Accuracy ─────────────────────────────────────────────
# [-1] gets the value from the last epoch
print(f"Training Accuracy   : {history.history['accuracy'][-1]:.4f}")
print(f"Validation Accuracy : {history.history['val_accuracy'][-1]:.4f}")

In [ ]:
# ── Step 10: Predict on Test Images ──────────────────────────────────────────
# argmax picks the class with the highest probability for each image
predictions     = model.predict(test_data)
predicted_classes = np.argmax(predictions, axis=1)
class_labels    = list(train_data.class_indices.keys())

print("Predictions:")
for i, cls in enumerate(predicted_classes):
    print(f"  Image {i+1}: {class_labels[cls]}")

In [ ]:
# ── Step 11: Plot Accuracy & Loss ────────────────────────────────────────────
plt.plot(history.history['accuracy'],     label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['loss'],     label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.show()